# ATLAS IMAGE AI: backend no Colab
1. **Ambiente de execução → Alterar o tipo → T4 GPU.**
2. **Executar tudo.** A segunda célula instala as dependências; a terceira baixa o backend oficial versionado em `backend/api.py`.
3. A última célula imprime o **link de conexão** (linha “ABRA NO CELULAR”). Abra no celular. Ela fica rodando de propósito: não pare.
4. O modelo baixa sob demanda. A primeira imagem pode demorar vários minutos; depois o tempo varia conforme modelo, GPU e cota do Colab.
5. Se rodar tudo de novo, use sempre o link novo. Não salve o notebook com a saída no GitHub: ela pode conter a chave de acesso.

In [ ]:
!pip -q install fastapi uvicorn diffusers transformers accelerate safetensors sentencepiece sacremoses bitsandbytes compel
import torch
assert torch.cuda.is_available(), "Sem GPU. Ative a T4 em Ambiente de execução → Alterar o tipo e execute de novo."
print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
# Backend fixado em commit para garantir uma versão reproduzível mesmo após a integração da branch.
import os
!wget -q -O atlas_api.py https://raw.githubusercontent.com/LucasMoraes023/atlas-image-ai/43b48fb4e6365a5639748c798f8efd2a632e859d/backend/api.py
assert os.path.getsize("atlas_api.py") > 5000, "Não foi possível baixar o backend oficial do ATLAS. Confira a conexão e execute novamente."
print("Backend ATLAS carregado:", os.path.getsize("atlas_api.py"), "bytes")

In [ ]:
import os, queue, re, secrets, subprocess, threading, time, urllib.request
FRONTEND_URL = "https://lucasmoraes023.github.io/atlas-image-ai/"  #@param {type:"string"}
LOCAL = "http://127.0.0.1:7860"

def log(m):
    print(f"[{time.strftime('%H:%M:%S')}] {m}", flush=True)

def up_local():
    try:
        urllib.request.urlopen(LOCAL + "/health", timeout=5)
        return True
    except Exception:
        return False

# encerra execuções anteriores (porta ocupada e chave antiga são causas comuns de "não conecta")
subprocess.run("pkill -f '[u]vicorn atlas_api'; pkill -f '[c]loudflared'; pkill -f '[l]ocalhost.run'", shell=True)
time.sleep(2)
KEY = secrets.token_urlsafe(24)
os.environ["ATLAS_KEY"] = KEY
logf = open("/content/atlas_server.log", "w")
log("1/3 Iniciando a API…")
subprocess.Popen(["python", "-m", "uvicorn", "atlas_api:app", "--host", "127.0.0.1", "--port", "7860", "--log-level", "warning"], stdout=logf, stderr=logf)
for _ in range(60):
    if up_local():
        break
    time.sleep(2)
else:
    print(open("/content/atlas_server.log").read()[-1500:])
    raise RuntimeError("A API não iniciou. Veja o log acima e execute a célula %%writefile de novo.")
log("API no ar.")
log("2/3 Criando o túnel público…")
subprocess.run(["wget", "-q", "-T", "30", "-O", "cloudflared", "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64"])
if os.path.exists("cloudflared"):
    os.chmod("cloudflared", 0o755)

def run_tunnel(cmd, pattern, wait=60):
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    found, tail = queue.Queue(), []
    def reader():
        for line in p.stdout:
            tail.append(line.strip()); del tail[:-8]
            m = re.search(pattern, line)
            if m and found.empty():
                found.put(m.group(0))
    threading.Thread(target=reader, daemon=True).start()
    try:
        return found.get(timeout=wait), tail
    except queue.Empty:
        p.kill()
        return None, tail

CF = (["./cloudflared", "tunnel", "--url", LOCAL, "--no-autoupdate"], r"https://[a-z0-9-]+\.trycloudflare\.com")
LHR = (["ssh", "-o", "StrictHostKeyChecking=no", "-o", "ServerAliveInterval=30", "-o", "ExitOnForwardFailure=yes", "-R", "80:127.0.0.1:7860", "nokey@localhost.run"], r"https://[a-z0-9]+\.lhr\.life")
url = None
for name, (cmd, pat) in [("Cloudflare", CF), ("Cloudflare, 2ª tentativa", CF), ("localhost.run", LHR)]:
    log(f"  tentando {name} (até 1 minuto)…")
    try:
        url, tail = run_tunnel(cmd, pat)
    except OSError as e:
        log(f"  indisponível: {e}")
        continue
    if url:
        break
    log("  falhou. Últimas linhas: " + " | ".join(tail[-3:]))
    time.sleep(5)
if not url:
    raise RuntimeError("Não consegui criar nenhum túnel. Espere 2 minutos e execute esta célula de novo; se persistir, envie as linhas acima.")
log("3/3 Pronto. O endereço novo pode levar até 1 minuto para funcionar no celular.")
print("\nABRA NO CELULAR (não compartilhe):\n" + FRONTEND_URL.rstrip("/") + "/#api=" + url + "&key=" + KEY)
print("\nMantenha esta célula rodando. A primeira imagem baixa o modelo (cerca de 7 GB) e pode levar de 5 a 15 minutos; depois, de 1 a 3 minutos por imagem.")

def predownload():
    try:
        from huggingface_hub import snapshot_download
        log("Baixando o modelo padrão em segundo plano…")
        snapshot_download("stabilityai/stable-diffusion-xl-base-1.0", allow_patterns=["*.json", "*.txt", "*.fp16.safetensors"])
        snapshot_download("Helsinki-NLP/opus-mt-ROMANCE-en")
        log("Modelo baixado. Pode gerar.")
    except Exception as e:
        log(f"Download em segundo plano falhou ({type(e).__name__}); o app baixa na primeira imagem.")

threading.Thread(target=predownload, daemon=True).start()
while True:
    time.sleep(60)